#### Exercice 1 : Trinité des Sommes et Audit de Robustesse ($T=4$)

Un robot de trading analyse un spread bivarié sur un échantillon glissant de $T = 4$ observations. Le processeur extrait les vecteurs de prix bruts et les prédictions théoriques MCO suivants :
*   Prix réels observés : $Y = [1.0, 3.0, 4.0, 8.0]$
*   Prédictions du modèle : $\hat{Y} = [1.5, 3.5, 4.5, 6.5]$

1.  **Calcul de la moyenne :** Déterminer la moyenne arithmétique empirique $\bar{y}$ de cet échantillon.
2.  **Décomposition de la variance :**
    *   a. Calculer la Somme des Carrés Totaux ($\text{TSS}$).
    *   b. Calculer la Somme des Carrés Expliqués ($\text{ESS}$).
    *   c. Calculer la Somme des Carrés des Résidus ($\text{RSS}$).
3.  **Vérification structurelle :** Valider l'identité fondamentale $\text{TSS} = \text{ESS} + \text{RSS}$. Le produit croisé s'annule-t-il parfaitement ?
4.  **Audit macroscopique :** En déduire la valeur exacte du coefficient de détermination ($R^2$). Le modèle valide-t-il le premier interrupteur automatique de production ($R^2 \ge 0.15$)?

In [20]:
import numpy as np

y_true = np.array([1.0, 3.0, 4.0, 8.0])
y_pred = np.array([1.5, 3.5, 4.5, 6.5])

y_mean = y_true.sum() / len(y_true)

print(f'y_mean = {y_mean}')

# Total Sum of Squares
tss = np.sum((y_true - y_mean) ** 2)
print(f'tss = {tss}')

# Explained Sum of Squares
ess = np.sum((y_pred - y_mean) ** 2)
print(f'ess = {ess}')

# Residual Sum of Squares
rss = np.sum((y_true - y_pred) ** 2)
print(f'rss = {rss}')

is_valid = np.allclose(tss, ess+rss)
print(f'tss == (ess + rss)? {"YES" if is_valid else "NO"}')
# This mean y_pred does not come from OLS (E and X not orthogonal)

# Note: this is different from ess/tss because y_pred is not generated from OLS,
# Otherwise, it would be equal. We use the original form instead since it measure
# how much we "captured error".
r2 = 1 - (rss / tss)
print(f'r2 = {r2:.4f}')

r2_treshold = 0.15
print(f"valid? {'YES' if r2 >= r2_treshold else 'NO'}")


y_mean = 4.0
tss = 26.0
ess = 13.0
rss = 3.0
tss == (ess + rss)? NO
r2 = 0.8846
valid? YES



#### Exercice 2 : Inférence Microscopique et Recherche Inverse

En poussant l'audit de l'échantillon précédent ($T=4$), l'ingénieur quantitatif souhaite cartographier les incertitudes du robot. Le calcul de la dispersion de l'indicateur directeur fournit la constante suivante : $\sum_{t=1}^{4} (x_t - \bar{x})^2 = 8.0$. La pente estimée par le modèle vaut $\hat{b} = 1.25$.

1.  **Estimation du bruit :** Calculer la variance résiduelle non biaisée du bruit du marché ($\hat{\sigma}_e^2$).
2.  **Unité d'incertitude :** Calculer la variance propre de la pente ($\sigma_{\hat{b}}^2$) puis son erreur standard ($\hat{\sigma}_{\hat{b}}$).
3.  **Inférence de Student :**
    *   a. Calculer la statistique $t$ de Student associée à cette pente.
    *   b. À l'aide de la table de Student, récupérer le seuil critique pour un risque $\alpha = 5\%$ en test bilatéral.
    *   c. Énoncer le verdict de décision probabiliste. Le robot doit-il activer ou désactiver ses modules d'ordres ?
4.  **Analyse de Puissance Inversée :** Si le modèle est rejeté, réalisez une recherche inverse sur la table de Student. À partir de combien de degrés de liberté ($df$) ce même score $t$ calculé aurait-il franchi la frontière critique ? En déduire la taille d'échantillon $T$ minimale pour valider cet Alpha.


In [38]:
sum_squared_x_dispersion = 8.0 # sum of squared dispersion (from mean)
b_hat = 1.25 # coef
T = 4

dof = 2
assert T > dof
sq_sigma_e = rss / (T - dof) # Don't forget to substract degrees-of-freedom
print(f'sq_sigma_e = {sq_sigma_e} (variance of residuals)')

sq_sigma_b = sq_sigma_e / sum_squared_x_dispersion
sigma_b = np.sqrt(sq_sigma_b)
print(f'sq_sigma_b = {sq_sigma_b:.4f} (variance of b_hat)')
print(f'sigma_b = {sigma_b:.4f}')

# Valeur centree reduite par rapport a l'hypothese null (b_hat = 0)
t_stat = b_hat / sigma_b
print(f't_stat = {t_stat:.4f}')

alpha = 0.05

from scipy import stats
# ppf (percent point function) use the "accumulated" value from the left,
# with is left_tail_surface + middle-surface = (alpha/2) + (1 - alpha).
# Which equal "1 - alpha / 2".
t_critical = stats.t.ppf(1.0 - (alpha / 2.0), df=dof)
print(f't_critical = {t_critical:.4f}')

student_test_valid = np.abs(t_stat) > t_critical
print(f"Student Test Valid? {
    'YES (H0 rejected => b_hat is solid)' if student_test_valid
    else 'NO (Cannot reject H0 => b_hat is uncertain)'}")

if not student_test_valid and t_stat > 1.9: # 1.9 is the threshold asymptote
    print('Would probably pass the test with more data (increased T)')


sq_sigma_e = 1.5 (variance of residuals)
sq_sigma_b = 0.1875 (variance of b_hat)
sigma_b = 0.4330
t_stat = 2.8868
t_critical = 4.3027
Student Test Valid? NO (Cannot reject H0 => b_hat is uncertain)
Would probably pass the test with more data (increased T)



#### Exercice 3 : Audit de Stabilité Horizontale et Confinement de l'Ancrage

Un robot Mean Reversion est déployé sur le spread EUR/USD vs GBP/USD avec une fenêtre historique de $T = 10$ heures. L'estimation fournit les métriques et moments d'ordre deux suivants :
*   Variance des résidus ($\hat{\sigma}_e^2$) = $0.0400$
*   Moyenne arithmétique de l'indicateur ($\bar{x}$) = $2.5$
*   Somme carrée centrée de l'indicateur $\sum_{t=1}^{10} (x_t - \bar{x})^2$ = $25.0$
*   Intercepte calculé ($\hat{a}$) = $0.12$

1.  **Calcul de l'incertitude pivot :** Calculer la variance théorique de l'intercepte ($\sigma_{\hat{a}}^2$) puis son erreur standard ($\hat{\sigma}_{\hat{a}}$) en appliquant la formule d'infrastructure de la section 6.6.4.
2.  **Test d'Ancrage (Approche 1) :** Calculer la statistique $t_a$ de Student pour l'intercepte. Sachant que le seuil critique pour $df = 8$ à $\alpha = 5\%$ vaut $t_{\text{critique}} = 2.306$, l'intercepte est-il statistiquement nul ? Quelle est la conséquence pour la ligne de base du bot ?
3.  **Audit Métrologique (Approche 2) :** Calculer le ratio de volatilité relative $\hat{\sigma}_{\hat{a}} / \hat{\sigma}_e$. En appliquant le seuil d'acceptation industriel de $15\%$, le modèle subit-il un glissement vertical inacceptable ou le pivot est-il déclaré sécurisé ?


In [55]:
T = 10
sq_sigma_e = 0.04
x_mean = 2.5
sum_sq_x_diff = 25.0
a_hat = 0.12 # intercept

sq_sigma_a = sq_sigma_e * ((1.0 / T) + ((x_mean ** 2) / sum_sq_x_diff))
sigma_a = np.sqrt(sq_sigma_a)
print(f'sq_sigma_a = {sq_sigma_a:.4f}')
print(f'sigma_a = {sigma_a:.4f}')

t_student = a_hat / sigma_a
t_critical = stats.t.ppf(1.0 - 0.05/2.0, df=(T-2))
print(f't_student = {t_student:.4f}')
print(f't_critical = {t_critical:.4f}')

pass_student_test = t_student >= t_critical
print(f"PASS: {'YES' if pass_student_test else 'NO'}")
# NO, a_hat is not significant enough
# Which is a "good thing" for Pair Traiding (=> spread is pure)

relative_volatility = sigma_a / np.sqrt(sq_sigma_e)
relative_threshold = 0.15
print(f'relative_volatility = {relative_volatility}')
print(f'relative_threshold = {relative_threshold}')
print(f"PASS: {'YES' if relative_volatility <= relative_threshold else 'NO'}")
# a_hat is unstable

sq_sigma_a = 0.0140
sigma_a = 0.1183
t_student = 1.0142
t_critical = 2.3060
PASS: NO
relative_volatility = 0.5916079783099615
relative_threshold = 0.15
PASS: NO



#### Exercice 4 : Optimisation de RAM et Dimensionnement de Rolling Window

Lors d'une phase de forte volatilité macroéconomique, un algorithme bivarié configuré par défaut sur une fenêtre glissante géante de $T = 1002$ bougies ($df = 1000$, $t_{\text{critique}} \approx 1.960$) extrait une corrélation linéaire extrême. Le score $t$ calculé pour la pente s'établit en direct à $\lvert t \rvert = 4.50$.

L'ingénieur quantitatif souhaite réduire au maximum la taille de cette fenêtre pour libérer de la mémoire vive et éliminer le retard temporel du bot.

1.  **Recherche Inverse :** En analysant l'extrait de la table de Student ci-dessous pour un risque $\alpha = 5\%$ bilatéral, identifiez à partir de quel nombre de degrés de liberté ($df$) la frontière critique devient inférieure ou égale à notre score calculé de $4.50$ :
    *   $df = 2 \implies t_{\text{critique}} = 4.303$
    *   $df = 3 \implies t_{\text{critique}} = 3.182$
    *   $df = 4 \implies t_{\text{critique}} = 2.776$
2.  **Sizing de Production :** En déduire la taille d'échantillon minimale $T_{\text{minimal}}$ exigée pour maintenir la certification statistique de ce signal.
3.  **Gain d'Infrastructure :** Quel est le pourcentage de réduction de taille de fenêtre obtenu ? Expliquez brièvement l'avantage de cette compression dynamique pour la réactivité du robot face aux changements de régime de marché.


In [47]:
T = 1002
coef_t_score = 4.50

df_min = T - 2

for t in range(1, T-2, 1):
    df = t - 2

    if df < 1:
        continue

    t_critical = stats.t.ppf(1.0 - 0.05/2.0, df=df)

    if t_critical <= coef_t_score:
        df_min = df
        break

t_min = df + 2
print(f'Sample size T={T} can be reduced to {t_min}')

reduction_pct = 100.0 * (1.0 - (t_min / T))
print(f'Sample windows reduced by {reduction_pct:.2f} %')

Sample size T=1002 can be reduced to 4
Sample windows reduced by 99.60 %


# Atelier

In [48]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np

project_root = os.path.abspath('/home/user/perso/trading/alphalab')

if project_root not in sys.path:
    sys.path.append(project_root)

In [60]:
from enl.bivariate_diagnostic import BivariateDiagnostic

if __name__ == "__main__":
    # Simulated validation dataset from TD Exercise 3:
    # T = 10, x_mean = 2.5, x_dispersion = 25.0
    x_test = np.array([1.0, 1.5, 2.0, 2.0, 2.5, 2.5, 3.0, 3.0, 3.5, 4.0])

    # Synthetic driven variable Y built to isolate intercept validation
    np.random.seed(42)
    base_y = 0.12 + 1.5 * x_test
    noise = np.array([-0.2, 0.15, -0.05, 0.22, -0.12, -0.18, 0.08, -0.02, 0.11, -0.01])
    y_test = base_y + noise * 0.932

    # Extract historical parameters using regular OLS formulas from Block 1.2
    x_mean_val = np.mean(x_test)
    y_mean_val = np.mean(y_test)
    beta_val = np.sum((x_test - x_mean_val) * (y_test - y_mean_val)) / np.sum((x_test - x_mean_val) ** 2)
    alpha_val = y_mean_val - beta_val * x_mean_val

    # Initialize the diagnostics engine with default target thresholds
    engine = BivariateDiagnostic(alpha=0.05, r2_min=0.15, ratio_anchor_max=0.15)

    # Execute the testing pipeline
    try:
        results = engine.audit_model(x_test, y_test, beta_val, alpha_val)

        print("=== QUANTITATIVE INFRASTRUCTURE AUDIT REPORT ===")
        print(f"Computed Slope (Hedge Ratio)      : {beta_val:.4f} (Expected: 1.5236)")
        print(f"Computed Intercept (Anchor Value)  : {alpha_val:.4f} (Expected: 0.0591)")
        print(f"Coefficient of Determination R²    : {results['metrics']['r2']:.4f} (Expected: 0.9914)")
        print(f"Slope Standard Error (sigma_b)    : {results['metrics']['sigma_b']:.4f} (Expected: 0.0502)")
        print(f"Anchor Standard Error (sigma_a)   : {results['metrics']['sigma_a']:.4f} (Expected: 0.1329)")
        print(f"Slope Student t-statistic         : {results['metrics']['t_stat_b']:.4f} (Expected: 30.3268)")
        print(f"Critical Boundary (t_critical)    : {results['metrics']['t_critical']:.4f} (Expected: 2.3060)")
        print(f"Anchor Metrological Flaw Ratio    : {results['metrics']['ratio_anchor']:.4f} (Expected: 0.9661)")
        print(f"Minimum Window Required (T_min)   : {results['metrics']['t_minimal_required']} (Expected: 3)")
        print("--- AUTOMATIC CIRCUIT BREAKERS INFRASTRUCTURE ---")
        print(f"Geometric R² Filter Validated     : {results['status']['circuit_breaker_r2']} (Expected: True)")
        print(f"Inference t-test Filter Validated : {results['status']['circuit_breaker_inference']} (Expected: True)")
        print(f"Metrological Anchor Pivot Healthy : {results['status']['circuit_breaker_anchor']} (Expected: False)")
        print(f"GLOBAL TRADING EXECUTION PASS TOKEN: {results['status']['robot_active']} (Expected: False)")

    except Exception as e:
        print(f"Pipeline crashed. Missing implementation logic: {str(e)}")

=== QUANTITATIVE INFRASTRUCTURE AUDIT REPORT ===
Computed Slope (Hedge Ratio)      : 1.5236 (Expected: 1.5236)
Computed Intercept (Anchor Value)  : 0.0591 (Expected: 0.0591)
Coefficient of Determination R²    : 0.9914 (Expected: 0.9914)
Slope Standard Error (sigma_b)    : 0.0502 (Expected: 0.0502)
Anchor Standard Error (sigma_a)   : 0.1329 (Expected: 0.1329)
Slope Student t-statistic         : 30.3268 (Expected: 30.3268)
Critical Boundary (t_critical)    : 2.3060 (Expected: 2.3060)
Anchor Metrological Flaw Ratio    : 0.9661 (Expected: 0.9661)
Minimum Window Required (T_min)   : 3 (Expected: 3)
--- AUTOMATIC CIRCUIT BREAKERS INFRASTRUCTURE ---
Geometric R² Filter Validated     : True (Expected: True)
Inference t-test Filter Validated : True (Expected: True)
Metrological Anchor Pivot Healthy : False (Expected: False)
GLOBAL TRADING EXECUTION PASS TOKEN: False (Expected: False)
